# SC-0-Cypherpunk-Origins - Les origines Cypherpunk de la blockchain

[Setup Foundry >>](SC-1-Setup-Foundry.ipynb)

---

## Objectifs d'apprentissage

1. Comprendre le **mouvement Cypherpunk** et son heritage technologique
2. Manipuler les **primitives cryptographiques** fondatrices (hash, signatures, PoW)
3. Construire une **mini-blockchain** a la main avec Python
4. Implementer un **arbre de Merkle** et comprendre son role
5. Decouvrir les **tables de hachage distribuees** (DHT/Kademlia)

### Prerequis

- Python 3.10+ avec `hashlib` (stdlib)
- `pycryptodome` pour les signatures numeriques
- Aucune connaissance blockchain prealable

### Duree estimee : 60 minutes

---

## 1. Le Manifeste Cypherpunk

> *"Privacy is necessary for an open society in the electronic age."*
> -- Eric Hughes, A Cypherpunk's Manifesto, 1993

Le mouvement Cypherpunk, ne dans les annees 1980-90, reunit des cryptographes, mathematiciens
et activistes convaincus que la **cryptographie** est l'outil fondamental de la liberte individuelle
a l'ere numerique.

### Textes fondateurs

| Annee | Auteur | Texte | Idee cle |
|-------|--------|-------|----------|
| 1985 | David Chaum | *Security without Identification* | Monnaie electronique anonyme |
| 1988 | Timothy May | *The Crypto Anarchist Manifesto* | Crypto = outil d'emancipation |
| 1993 | Eric Hughes | *A Cypherpunk's Manifesto* | "Cypherpunks write code" |
| 1997 | Adam Back | Hashcash | Proof-of-Work anti-spam |
| 1998 | Wei Dai | b-money | Monnaie decentralisee theorique |
| 2008 | Satoshi Nakamoto | Bitcoin whitepaper | Synthese de toutes ces idees |

Les smart contracts modernes (Ethereum, 2015) sont l'**aboutissement direct** de 30 ans
d'innovations Cypherpunk. Ce notebook retrace ces briques fondatrices avec du **vrai code executable**.

In [1]:
# Filiation technologique : des Cypherpunks aux Smart Contracts
filiation = {
    "1991 - PGP (Zimmermann)":        "Chiffrement asymetrique grand public",
    "1997 - Hashcash (Back)":          "Proof-of-Work -> minage Bitcoin",
    "1998 - b-money (Dai)":            "Monnaie decentralisee theorique",
    "1999 - Napster":                  "P2P massif (centralise) -> BitTorrent",
    "2001 - BitTorrent (Cohen)":       "DHT Kademlia -> reseau Ethereum",
    "2004 - RPoW (Finney)":            "Proof-of-Work reutilisable",
    "2008 - Bitcoin (Nakamoto)":       "Hash chain + PoW + P2P + signatures",
    "2013 - Ethereum (Buterin)":       "Bitcoin + Turing-completude = Smart Contracts",
}

print("FILIATION CYPHERPUNK -> SMART CONTRACTS")
print("=" * 60)
for event, contribution in filiation.items():
    print(f"  {event}")
    print(f"    -> {contribution}")
    print()

FILIATION CYPHERPUNK -> SMART CONTRACTS
  1991 - PGP (Zimmermann)
    -> Chiffrement asymetrique grand public

  1997 - Hashcash (Back)
    -> Proof-of-Work -> minage Bitcoin

  1998 - b-money (Dai)
    -> Monnaie decentralisee theorique

  1999 - Napster
    -> P2P massif (centralise) -> BitTorrent

  2001 - BitTorrent (Cohen)
    -> DHT Kademlia -> reseau Ethereum

  2004 - RPoW (Finney)
    -> Proof-of-Work reutilisable

  2008 - Bitcoin (Nakamoto)
    -> Hash chain + PoW + P2P + signatures

  2013 - Ethereum (Buterin)
    -> Bitcoin + Turing-completude = Smart Contracts



### Observation

Chaque innovation de cette filiation repose sur un **petit nombre de primitives cryptographiques** :
le hachage, les signatures numeriques, la preuve de travail, et les reseaux pair-a-pair.
Bitcoin n'a rien invente : il a **combine** ces briques existantes de maniere geniale.

Les sections suivantes explorent chacune de ces primitives avec du code executable.

---

## 2. Hash et integrite

Le **hachage cryptographique** est la brique la plus fondamentale. Une fonction de hachage
transforme une donnee de taille arbitraire en une empreinte de taille fixe, avec trois proprietes :

1. **Determinisme** : meme entree -> meme sortie, toujours
2. **Effet avalanche** : 1 bit change -> sortie completement differente
3. **Resistance aux collisions** : quasi-impossible de trouver deux entrees avec le meme hash

SHA-256 (Secure Hash Algorithm, 256 bits) est utilise par Bitcoin et de nombreuses blockchains.

In [2]:
import hashlib

# SHA-256 : la brique fondamentale de Bitcoin
data = b"Cypherpunks write code"
hash_value = hashlib.sha256(data).hexdigest()
print(f"Message  : {data.decode()}")
print(f"SHA-256  : {hash_value}")
print(f"Longueur : {len(hash_value)} caracteres hex = {len(hash_value)*4} bits")
print()

# Effet avalanche : un seul caractere change
data2 = b"Cypherpunks write Code"  # 'c' -> 'C'
hash2 = hashlib.sha256(data2).hexdigest()
print(f"Message  : {data2.decode()}")
print(f"SHA-256  : {hash2}")
print()

# Compter les bits differents (distance de Hamming)
diff_bits = bin(int(hash_value, 16) ^ int(hash2, 16)).count('1')
print(f"Bits differents : {diff_bits} / 256 ({diff_bits/256*100:.1f}%)")
print("-> Un seul caractere change, ~50% des bits changent (effet avalanche)")

Message  : Cypherpunks write code
SHA-256  : 42cc22190b177e5c48e32fe87c214d88eb21cac7780aad65b8b816d77cf22820
Longueur : 64 caracteres hex = 256 bits

Message  : Cypherpunks write Code
SHA-256  : b211556e694dbc8f1c6d46e438d89cb1d99c285489c066282fdd5dfb542a646d

Bits differents : 130 / 256 (50.8%)
-> Un seul caractere change, ~50% des bits changent (effet avalanche)


### Interpretation

L'**effet avalanche** est crucial pour la securite : il est impossible de deviner le hash
a partir de petites modifications de l'entree. En pratique, ~128 bits sur 256 changent
(~50%), ce qui confirme que SHA-256 se comporte comme une fonction aleatoire.

Cette propriete est exploitee dans :
- **Bitcoin** : le hash du bloc doit commencer par N zeros (Proof-of-Work)
- **Ethereum** : Keccak-256 pour les adresses et les signatures
- **Git** : SHA-1 pour identifier les commits (SHA-256 en migration)

---

## 3. Chaine de hash (proto-blockchain)

L'idee centrale de Bitcoin est de **chainer les blocs par leur hash** :
chaque bloc contient le hash du bloc precedent. Modifier un ancien bloc
invalide tous les blocs suivants.

Ce concept existait deja dans les travaux de Stuart Haber et Scott Stornetta (1991)
sur l'horodatage de documents numeriques.

In [3]:
import hashlib
import json
from datetime import datetime

def create_block(index, data, previous_hash):
    """Creer un bloc avec lien vers le precedent via hash."""
    block = {
        "index": index,
        "timestamp": datetime.now().isoformat(),
        "data": data,
        "previous_hash": previous_hash,
    }
    block_string = json.dumps(block, sort_keys=True).encode()
    block["hash"] = hashlib.sha256(block_string).hexdigest()
    return block

# Construire une mini-blockchain
genesis = create_block(0, "Bloc Genesis", "0" * 64)
block1 = create_block(1, "Alice envoie 10 BTC a Bob", genesis["hash"])
block2 = create_block(2, "Bob envoie 5 BTC a Charlie", block1["hash"])
block3 = create_block(3, "Charlie envoie 2 BTC a Dave", block2["hash"])

chain = [genesis, block1, block2, block3]

print("MINI-BLOCKCHAIN (4 blocs)")
print("=" * 70)
for block in chain:
    print(f"Bloc {block['index']} : {block['data']}")
    print(f"  Hash     : {block['hash'][:32]}...")
    print(f"  Prev     : {block['previous_hash'][:32]}...")
    print()

MINI-BLOCKCHAIN (4 blocs)
Bloc 0 : Bloc Genesis
  Hash     : efe92349242974774cde81e9a913a05b...
  Prev     : 00000000000000000000000000000000...

Bloc 1 : Alice envoie 10 BTC a Bob
  Hash     : 33b96429d24d4a2055afd9ef9a31bd8b...
  Prev     : efe92349242974774cde81e9a913a05b...

Bloc 2 : Bob envoie 5 BTC a Charlie
  Hash     : 5a99bf8ebe73dcb2392618c070c87812...
  Prev     : 33b96429d24d4a2055afd9ef9a31bd8b...

Bloc 3 : Charlie envoie 2 BTC a Dave
  Hash     : bb72ec507efd77b9a39a7f9f9b6b4fe5...
  Prev     : 5a99bf8ebe73dcb2392618c070c87812...



Détection de falsification dans la chaîne de blocs en vérifiant l'intégrité des hash successifs.

In [4]:
# Detection de falsification
print("DETECTION DE FALSIFICATION")
print("=" * 70)

original_data = chain[1]["data"]
original_hash = chain[1]["hash"]

# Modifier le bloc 1 (falsification)
chain[1]["data"] = "Alice envoie 1000 BTC a Bob"  # Fraude !
block_string = json.dumps(
    {k: v for k, v in chain[1].items() if k != "hash"}, sort_keys=True
).encode()
new_hash = hashlib.sha256(block_string).hexdigest()

print(f"Bloc 1 original : '{original_data}'")
print(f"Bloc 1 modifie  : '{chain[1]['data']}'")
print()
print(f"Hash original   : {original_hash[:40]}...")
print(f"Hash recalcule  : {new_hash[:40]}...")
print()

# Verification de la chaine
print("VERIFICATION DE LA CHAINE :")
chain[1]["hash"] = new_hash
for i in range(1, len(chain)):
    expected_prev = chain[i-1]["hash"]
    actual_prev = chain[i]["previous_hash"]
    valid = expected_prev == actual_prev
    status = "OK" if valid else "INVALIDE"
    print(f"  Bloc {i} -> previous_hash {status}")
    if not valid:
        print(f"    Attendu : {expected_prev[:32]}...")
        print(f"    Trouve  : {actual_prev[:32]}...")

# Restaurer
chain[1]["data"] = original_data
chain[1]["hash"] = original_hash
print()
print("-> Le bloc 2 pointe vers l'ancien hash du bloc 1 : la fraude est detectee !")
print("-> Pour falsifier, il faudrait recalculer TOUS les blocs suivants.")

DETECTION DE FALSIFICATION
Bloc 1 original : 'Alice envoie 10 BTC a Bob'
Bloc 1 modifie  : 'Alice envoie 1000 BTC a Bob'

Hash original   : 33b96429d24d4a2055afd9ef9a31bd8b03db0254...
Hash recalcule  : 1c08d21447728108e05ebe07824e5b9a0a9f2f61...

VERIFICATION DE LA CHAINE :
  Bloc 1 -> previous_hash OK
  Bloc 2 -> previous_hash INVALIDE
    Attendu : 1c08d21447728108e05ebe07824e5b9a...
    Trouve  : 33b96429d24d4a2055afd9ef9a31bd8b...
  Bloc 3 -> previous_hash OK

-> Le bloc 2 pointe vers l'ancien hash du bloc 1 : la fraude est detectee !
-> Pour falsifier, il faudrait recalculer TOUS les blocs suivants.


### Interpretation

La chaine de hash cree une **structure de donnees immuable** : modifier un bloc ancien
casse tous les liens subsequents. C'est le fondement de l'integrite blockchain.

Mais cela ne suffit pas : un attaquant pourrait recalculer toute la chaine.
C'est la que la **Proof-of-Work** intervient (section 5) : rendre le recalcul
prohibitivement couteux en temps et en energie.

---

## 4. Arbre de Merkle

Un **arbre de Merkle** (Ralph Merkle, 1979) permet de verifier l'integrite de N elements
avec seulement O(log N) hash. Il est utilise dans :

- **Bitcoin** : verifier qu'une transaction est dans un bloc (SPV)
- **BitTorrent** : verifier l'integrite des pieces telechargees
- **Git** : la structure interne des commits
- **Ethereum** : Patricia Merkle Trie pour l'etat des comptes

### Principe

```
        Root Hash
       /         \\
    H(AB)       H(CD)
    /   \\       /   \\
  H(A)  H(B)  H(C)  H(D)
   |     |     |     |
  Tx A  Tx B  Tx C  Tx D
```

Pour prouver que Tx B est dans l'arbre, il suffit de fournir : H(A) + H(CD) + Root.

In [5]:
import hashlib

def sha256(data):
    """Hash SHA-256 d'une chaine."""
    if isinstance(data, str):
        data = data.encode()
    return hashlib.sha256(data).hexdigest()

def merkle_tree(transactions):
    """Construire un arbre de Merkle complet et retourner tous les niveaux."""
    if not transactions:
        return []
    current_level = [sha256(tx) for tx in transactions]
    tree = [current_level[:]]

    while len(current_level) > 1:
        next_level = []
        for i in range(0, len(current_level), 2):
            left = current_level[i]
            right = current_level[i + 1] if i + 1 < len(current_level) else left
            parent = sha256(left + right)
            next_level.append(parent)
        current_level = next_level
        tree.append(current_level[:])
    return tree

# Transactions exemple (comme dans un bloc Bitcoin)
transactions = [
    "Alice -> Bob: 1 BTC",
    "Bob -> Charlie: 0.5 BTC",
    "Dave -> Eve: 2 BTC",
    "Eve -> Frank: 0.3 BTC",
]

tree = merkle_tree(transactions)

print("ARBRE DE MERKLE")
print("=" * 70)
for level_idx, level in enumerate(tree):
    if level_idx == 0:
        label = "Feuilles"
    elif level_idx == len(tree) - 1:
        label = "Racine"
    else:
        label = f"Niveau {level_idx}"

    print(f"\n{label} ({len(level)} noeud(s)) :")
    for j, h in enumerate(level):
        if level_idx == 0:
            print(f"  [{j}] {h[:16]}... <- '{transactions[j]}'")
        else:
            print(f"  [{j}] {h[:16]}...")

print(f"\nMerkle Root : {tree[-1][0]}")
print(f"-> 4 transactions resumees en 1 seul hash de 256 bits")

ARBRE DE MERKLE

Feuilles (4 noeud(s)) :
  [0] 7d647288b6736b19... <- 'Alice -> Bob: 1 BTC'
  [1] b402cd80223159c9... <- 'Bob -> Charlie: 0.5 BTC'
  [2] 76b50d083bbe5124... <- 'Dave -> Eve: 2 BTC'
  [3] c9f9c448a2d7282f... <- 'Eve -> Frank: 0.3 BTC'

Niveau 1 (2 noeud(s)) :
  [0] 20e67cf979e88cda...
  [1] 613dc66e12a2097f...

Racine (1 noeud(s)) :
  [0] 8d1a98764d1baf07...

Merkle Root : 8d1a98764d1baf079ed3bf427789bbf1e9454a51c9c2be546cb487ec1c9ad78e
-> 4 transactions resumees en 1 seul hash de 256 bits


Génération d'une preuve Merkle permettant de vérifier qu'une transaction appartient à un bloc sans télécharger toute la blockchain.

In [6]:
def merkle_proof(transactions, tx_index):
    """Generer la preuve Merkle pour une transaction donnee."""
    tree = merkle_tree(transactions)
    proof = []
    idx = tx_index
    for level in tree[:-1]:
        if idx % 2 == 0:
            sibling_idx = idx + 1 if idx + 1 < len(level) else idx
            proof.append(("right", level[sibling_idx]))
        else:
            proof.append(("left", level[idx - 1]))
        idx //= 2
    return proof

def verify_proof(tx, proof, expected_root):
    """Verifier une preuve Merkle."""
    current = sha256(tx)
    for side, sibling in proof:
        if side == "right":
            current = sha256(current + sibling)
        else:
            current = sha256(sibling + current)
    return current == expected_root

# Generer et verifier la preuve pour la transaction 1
tx_index = 1
proof = merkle_proof(transactions, tx_index)
root = tree[-1][0]

print(f"PREUVE MERKLE pour transaction [{tx_index}]: '{transactions[tx_index]}'")
print("=" * 70)
print(f"Merkle Root attendue : {root[:32]}...")
print(f"\nPreuve ({len(proof)} elements, au lieu de {len(transactions)} transactions) :")
for side, h in proof:
    print(f"  {side:5s} : {h[:32]}...")

valid = verify_proof(transactions[tx_index], proof, root)
print(f"\nVerification : {'VALIDE' if valid else 'INVALIDE'}")
print(f"\n-> SPV (Bitcoin) : un noeud leger verifie une transaction")
print(f"   avec seulement {len(proof)} hash au lieu de {len(transactions)} transactions")

PREUVE MERKLE pour transaction [1]: 'Bob -> Charlie: 0.5 BTC'
Merkle Root attendue : 8d1a98764d1baf079ed3bf427789bbf1...

Preuve (2 elements, au lieu de 4 transactions) :
  left  : 7d647288b6736b19ab3bf3074249dea5...
  right : 613dc66e12a2097fd4a966c052140615...

Verification : VALIDE

-> SPV (Bitcoin) : un noeud leger verifie une transaction
   avec seulement 2 hash au lieu de 4 transactions


### Interpretation

L'arbre de Merkle permet la **verification legere** (SPV) : un telephone mobile
peut verifier qu'une transaction est incluse dans un bloc Bitcoin sans telecharger
les ~500 000 blocs complets (~500 Go).

| Transactions dans le bloc | Hash necessaires pour la preuve |
|--------------------------|-------------------------------|
| 1 000 | 10 |
| 1 000 000 | 20 |
| 1 000 000 000 | 30 |

La complexite logarithmique rend le systeme scalable a l'echelle mondiale.

---

## 5. Proof-of-Work (Hashcash)

**Adam Back** a invente Hashcash en 1997 comme systeme anti-spam pour les emails :
pour envoyer un email, il faut trouver un nonce tel que le hash du message commence
par N zeros. Cela coute quelques secondes de calcul a l'expediteur, mais rend
l'envoi massif de spam economiquement impossible.

Satoshi Nakamoto a directement repris ce mecanisme pour le **minage Bitcoin** :
- Le mineur cherche un nonce tel que `SHA256(bloc + nonce)` commence par N zeros
- La **difficulte** N s'ajuste toutes les ~2 semaines pour maintenir ~10 min/bloc
- La difficulte croit **exponentiellement** : chaque zero supplementaire double le travail moyen

In [7]:
import hashlib
import time

def proof_of_work(data, difficulty):
    """Trouver un nonce tel que hash(data+nonce) commence par N zeros."""
    target = "0" * difficulty
    nonce = 0
    start = time.time()
    while True:
        attempt = f"{data}{nonce}".encode()
        hash_val = hashlib.sha256(attempt).hexdigest()
        if hash_val[:difficulty] == target:
            elapsed = time.time() - start
            return nonce, hash_val, elapsed
        nonce += 1

data = "Bloc #42 - Alice envoie 1 BTC a Bob - prev_hash=abc123"

print("PROOF-OF-WORK (Hashcash / Bitcoin Mining)")
print("=" * 70)
print(f"Donnees : '{data[:50]}...'")
print()

for difficulty in range(1, 6):
    nonce, hash_val, elapsed = proof_of_work(data, difficulty)
    print(f"Difficulte {difficulty} (cible: {'0'*difficulty}{'x'*(8-difficulty)}) :")
    print(f"  Nonce    : {nonce:>10,}")
    print(f"  Hash     : {hash_val[:32]}...")
    print(f"  Temps    : {elapsed:.4f}s")
    print(f"  Essais   : ~{nonce:,} (theorique: ~{16**difficulty:,})")
    print()

PROOF-OF-WORK (Hashcash / Bitcoin Mining)
Donnees : 'Bloc #42 - Alice envoie 1 BTC a Bob - prev_hash=ab...'

Difficulte 1 (cible: 0xxxxxxx) :
  Nonce    :          6
  Hash     : 0c9aea00f1df55b854246da164d2cc73...
  Temps    : 0.0000s
  Essais   : ~6 (theorique: ~16)

Difficulte 2 (cible: 00xxxxxx) :
  Nonce    :        114
  Hash     : 003ecdb4d0fffd8a942d7ef196fbdba1...
  Temps    : 0.0001s
  Essais   : ~114 (theorique: ~256)

Difficulte 3 (cible: 000xxxxx) :
  Nonce    :        410
  Hash     : 000fcfcc32f345988c3f635964699b0b...
  Temps    : 0.0003s
  Essais   : ~410 (theorique: ~4,096)

Difficulte 4 (cible: 0000xxxx) :
  Nonce    :     39,430
  Hash     : 000034fef981e1dc59961bd97dcd6c69...
  Temps    : 0.0329s
  Essais   : ~39,430 (theorique: ~65,536)



Difficulte 5 (cible: 00000xxx) :
  Nonce    :    214,484
  Hash     : 00000bb794a36ce9ac55025613a1a499...
  Temps    : 0.1742s
  Essais   : ~214,484 (theorique: ~1,048,576)



### Interpretation

| Difficulte | Essais moyens | Temps approximatif |
|-----------|--------------|-------------------|
| 1 | ~16 | instantane |
| 2 | ~256 | instantane |
| 3 | ~4 096 | quelques ms |
| 4 | ~65 536 | ~0.05s |
| 5 | ~1 048 576 | ~0.5s |
| 6 | ~16 millions | ~10s |

La croissance est **exponentielle** (x16 par niveau). Bitcoin ajuste la difficulte pour que
le reseau mondial (~500 EH/s en 2025) mette ~10 minutes par bloc.

C'est ce mecanisme qui rend la falsification de la chaine de hash (section 3)
**economiquement impossible** : recalculer les blocs modifies prendrait plus d'energie
que l'ensemble du reseau honnete.

---

## 6. Signatures numeriques

Les signatures numeriques sont l'equivalent electronique de la signature manuscrite,
mais avec des garanties mathematiques :

1. **Authentification** : seul le detenteur de la cle privee peut signer
2. **Integrite** : toute modification du message invalide la signature
3. **Non-repudiation** : le signataire ne peut nier avoir signe

Bitcoin et Ethereum utilisent des signatures sur **courbes elliptiques** (ECDSA/secp256k1).
C'est le mecanisme qui lie une adresse (cle publique) a une transaction.

In [8]:
from Crypto.PublicKey import ECC
from Crypto.Signature import DSS
from Crypto.Hash import SHA256

print("SIGNATURES NUMERIQUES (Courbes Elliptiques)")
print("=" * 70)

# Generation de cles (equivalent d'un wallet crypto)
key = ECC.generate(curve='P-256')
public_key = key.public_key()

print(f"Cle privee (secret) : {hex(key.d)[:32]}...")
print(f"Cle publique X      : {hex(public_key.pointQ.x)[:32]}...")
print(f"Cle publique Y      : {hex(public_key.pointQ.y)[:32]}...")
print()

# Signer une transaction
transaction = b"Envoyer 1 ETH de 0xAlice a 0xBob"
h = SHA256.new(transaction)
signer = DSS.new(key, 'fips-186-3')
signature = signer.sign(h)

print(f"Transaction : {transaction.decode()}")
print(f"Signature   : {signature.hex()[:64]}...")
print(f"Taille      : {len(signature)} octets")
print()

# Verification (n'importe qui peut verifier avec la cle publique)
verifier = DSS.new(public_key, 'fips-186-3')
try:
    verifier.verify(SHA256.new(transaction), signature)
    print("Verification : VALIDE (la transaction est authentique)")
except ValueError:
    print("Verification : INVALIDE")

# Tentative de falsification
print()
fake_transaction = b"Envoyer 100 ETH de 0xAlice a 0xBob"
try:
    verifier.verify(SHA256.new(fake_transaction), signature)
    print("Falsification : ACCEPTEE (PROBLEME !)")
except ValueError:
    print("Falsification : REJETEE (la signature ne correspond pas)")
    print("-> Impossible de modifier la transaction sans la cle privee")

SIGNATURES NUMERIQUES (Courbes Elliptiques)
Cle privee (secret) : 0x61cd991a8c3d06b0454e4e215c0be6...
Cle publique X      : 0x59c7109b95b77edb210c62456b0265...
Cle publique Y      : 0xc5245843ee38ee3a9509d308284809...

Transaction : Envoyer 1 ETH de 0xAlice a 0xBob
Signature   : ae129c00076ee0aeda93f4148f1f80430571e3d83cdb5d0205bee423ef788966...
Taille      : 64 octets

Verification : VALIDE (la transaction est authentique)

Falsification : REJETEE (la signature ne correspond pas)
-> Impossible de modifier la transaction sans la cle privee


### Interpretation

Le systeme de signatures ECDSA garantit que :
- Seul Alice (detentrice de la cle privee) peut autoriser un transfert depuis son adresse
- Modifier le montant ou le destinataire invalide la signature
- N'importe quel noeud du reseau peut verifier la signature avec la cle publique

**Adresse blockchain = hash de la cle publique**

```
Cle privee (256 bits, secret)
    | multiplication sur courbe elliptique
Cle publique (512 bits, public)
    | SHA-256 + RIPEMD-160
Adresse (160 bits = 20 octets = "0x...")
```

C'est pourquoi perdre sa cle privee = perdre ses fonds, et pourquoi il ne faut **jamais** la partager.

---

## 7. DHT et reseaux pair-a-pair

Le dernier ingredient est le **reseau decentralise**. Sans serveur central,
comment les noeuds Bitcoin/Ethereum se trouvent-ils et echangent-ils des donnees ?

La reponse vient de **BitTorrent** et du protocole **Kademlia** (2002) :
une **table de hachage distribuee** (DHT) ou chaque noeud stocke une partie des donnees,
et la distance entre noeuds est mesuree par l'operation **XOR** sur leurs identifiants.

Ethereum utilise directement une variante de Kademlia pour la decouverte de pairs.

In [9]:
import hashlib

# Chaque noeud du reseau a un identifiant = hash de son adresse
nodes = {
    "noeud-paris:30303":   int(hashlib.sha256(b"noeud-paris:30303").hexdigest(), 16),
    "noeud-london:30303":  int(hashlib.sha256(b"noeud-london:30303").hexdigest(), 16),
    "noeud-tokyo:30303":   int(hashlib.sha256(b"noeud-tokyo:30303").hexdigest(), 16),
    "noeud-nyc:30303":     int(hashlib.sha256(b"noeud-nyc:30303").hexdigest(), 16),
}

print("TABLE DE HACHAGE DISTRIBUEE (DHT / Kademlia)")
print("=" * 70)

print("\nIdentifiants des noeuds (SHA-256 tronque a 16 hex) :")
for name, node_id in nodes.items():
    print(f"  {name:25s} -> {node_id & 0xFFFFFFFFFFFFFFFF:016x}")

# Distance XOR entre noeuds
print("\nDistances XOR (plus petit = plus proche) :")
names = list(nodes.keys())
ids = list(nodes.values())

for i in range(len(names)):
    for j in range(i + 1, len(names)):
        distance = ids[i] ^ ids[j]
        dist_bits = distance.bit_length()
        print(f"  {names[i]:15s} <-> {names[j]:15s} : {dist_bits} bits")

print()
print("-> Kademlia : chaque noeud connait ~log(N) autres noeuds")
print("-> Trouver une donnee prend ~log(N) sauts (routage iteratif)")
print("-> Pas de serveur central, resilient aux pannes et a la censure")

TABLE DE HACHAGE DISTRIBUEE (DHT / Kademlia)

Identifiants des noeuds (SHA-256 tronque a 16 hex) :
  noeud-paris:30303         -> 3dbba68547a2f49d
  noeud-london:30303        -> d8c971c6e2920124
  noeud-tokyo:30303         -> 5554c6f0b2c809db
  noeud-nyc:30303           -> bfa57edb7fb9d6f9

Distances XOR (plus petit = plus proche) :
  noeud-paris:30303 <-> noeud-london:30303 : 256 bits
  noeud-paris:30303 <-> noeud-tokyo:30303 : 256 bits
  noeud-paris:30303 <-> noeud-nyc:30303 : 255 bits
  noeud-london:30303 <-> noeud-tokyo:30303 : 254 bits
  noeud-london:30303 <-> noeud-nyc:30303 : 256 bits
  noeud-tokyo:30303 <-> noeud-nyc:30303 : 256 bits

-> Kademlia : chaque noeud connait ~log(N) autres noeuds
-> Trouver une donnee prend ~log(N) sauts (routage iteratif)
-> Pas de serveur central, resilient aux pannes et a la censure


Implémentation du format Bencode utilisé par BitTorrent pour sérialiser les métadonnées de torrents.

In [10]:
import hashlib

def bencode(data):
    """Encoder des donnees au format bencode (BitTorrent)."""
    if isinstance(data, int):
        return f"i{data}e".encode()
    elif isinstance(data, bytes):
        return f"{len(data)}:".encode() + data
    elif isinstance(data, str):
        data_bytes = data.encode()
        return f"{len(data_bytes)}:".encode() + data_bytes
    elif isinstance(data, list):
        return b"l" + b"".join(bencode(x) for x in data) + b"e"
    elif isinstance(data, dict):
        items = sorted(data.items())
        return b"d" + b"".join(
            bencode(k.encode() if isinstance(k, str) else k) + bencode(v)
            for k, v in items
        ) + b"e"
    raise TypeError(f"Type non supporte: {type(data)}")

# Simuler un fichier .torrent
torrent_info = {
    "name": "bitcoin-whitepaper.pdf",
    "length": 184292,
    "piece length": 262144,
    "pieces": b"\x00" * 20,
}

encoded = bencode(torrent_info)
info_hash = hashlib.sha1(encoded).hexdigest()

print("BENCODAGE BITTORRENT")
print("=" * 70)
print(f"Fichier     : {torrent_info['name']}")
print(f"Taille      : {torrent_info['length']:,} octets")
print(f"Bencode     : {encoded[:60]}...")
print(f"Info hash   : {info_hash}")
print()
print(f"-> L'info_hash est l'identifiant unique du torrent dans la DHT")
print(f"-> C'est le 'magnet link' : magnet:?xt=urn:btih:{info_hash[:20]}...")

BENCODAGE BITTORRENT
Fichier     : bitcoin-whitepaper.pdf
Taille      : 184,292 octets
Bencode     : b'd6:lengthi184292e4:name22:bitcoin-whitepaper.pdf12:piece len'...
Info hash   : 744ab5f5c309dcae224fb2c44f3591cd7ae2671f

-> L'info_hash est l'identifiant unique du torrent dans la DHT
-> C'est le 'magnet link' : magnet:?xt=urn:btih:744ab5f5c309dcae224f...


### Interpretation

Les reseaux P2P apportent la **decentralisation** necessaire aux blockchains :

| Propriete | Serveur central | P2P (DHT) |
|-----------|----------------|----------|
| Point de defaillance unique | Oui | Non |
| Censurable | Oui (saisie du serveur) | Tres difficile |
| Scalabilite | Limitee | O(log N) par noeud |
| Exemples | Napster (1999, ferme) | BitTorrent (2001, toujours actif) |

Ethereum herite directement de Kademlia : le port par defaut (30303) et le protocole
de decouverte sont bases sur la DHT.

---

## 8. Synthese : de Chaum a Ethereum

Voici la frise complete montrant comment chaque brique Cypherpunk
a ete integree dans les blockchains modernes.

In [11]:
# Frise chronologique : briques fondatrices et leur heritage
timeline = [
    (1976, "Diffie-Hellman",     "Echange de cles",           ["Signature"], "CRYPTO"),
    (1977, "RSA",                "Chiffrement asymetrique",   ["Signature"], "CRYPTO"),
    (1979, "Merkle",             "Arbre de Merkle",           ["Bitcoin SPV", "Git", "BitTorrent"], "STRUCTURE"),
    (1985, "Chaum",              "eCash (monnaie anonyme)",   ["Zcash", "Monero"], "MONNAIE"),
    (1991, "PGP",                "Crypto grand public",       ["GPG", "Signal"], "CRYPTO"),
    (1991, "Haber-Stornetta",    "Horodatage par hash chain", ["Bitcoin blockchain"], "STRUCTURE"),
    (1997, "Hashcash",           "Proof-of-Work",             ["Bitcoin mining"], "CONSENSUS"),
    (1998, "b-money",            "Monnaie decentralisee",     ["Bitcoin"], "MONNAIE"),
    (2001, "BitTorrent",         "DHT Kademlia, P2P massif",  ["Ethereum P2P"], "RESEAU"),
    (2002, "Kademlia",           "DHT avec distance XOR",     ["Ethereum devp2p"], "RESEAU"),
    (2008, "Bitcoin",            "Synthese de tout",          ["Ethereum", "1000+ altcoins"], "BLOCKCHAIN"),
    (2013, "Ethereum",           "Smart Contracts (Turing)",  ["DeFi", "NFT", "DAO"], "BLOCKCHAIN"),
    (2015, "Ethereum mainnet",   "Lancement production",      ["ERC-20", "ERC-721", "DeFi 2020"], "BLOCKCHAIN"),
]

print("FRISE CHRONOLOGIQUE : CYPHERPUNKS -> SMART CONTRACTS")
print("=" * 70)

for year, name, description, heritage, cat in timeline:
    marker = {"CRYPTO": "[C]", "STRUCTURE": "[S]", "MONNAIE": "[M]",
              "CONSENSUS": "[W]", "RESEAU": "[R]", "BLOCKCHAIN": "[B]"}[cat]
    print(f"  {year} {marker} {name:20s} | {description}")
    if heritage:
        print(f"       {'':20s}   -> {', '.join(heritage)}")
    print()

print("Legende : [C]=Crypto [S]=Structure [M]=Monnaie [W]=Consensus [R]=Reseau [B]=Blockchain")

FRISE CHRONOLOGIQUE : CYPHERPUNKS -> SMART CONTRACTS
  1976 [C] Diffie-Hellman       | Echange de cles
                              -> Signature

  1977 [C] RSA                  | Chiffrement asymetrique
                              -> Signature

  1979 [S] Merkle               | Arbre de Merkle
                              -> Bitcoin SPV, Git, BitTorrent

  1985 [M] Chaum                | eCash (monnaie anonyme)
                              -> Zcash, Monero

  1991 [C] PGP                  | Crypto grand public
                              -> GPG, Signal

  1991 [S] Haber-Stornetta      | Horodatage par hash chain
                              -> Bitcoin blockchain

  1997 [W] Hashcash             | Proof-of-Work
                              -> Bitcoin mining

  1998 [M] b-money              | Monnaie decentralisee
                              -> Bitcoin

  2001 [R] BitTorrent           | DHT Kademlia, P2P massif
                              -> Ethereum P2P

  2002 [R] Kademli

---

## 9. Exercice : Construire une mini-blockchain complete

En combinant toutes les briques de ce notebook, implementez une mini-blockchain fonctionnelle.

In [12]:
# Exercice : Mini-blockchain complete
# Combiner : hash chain + Merkle tree + PoW + signatures

import hashlib
import json
import time
from Crypto.PublicKey import ECC
from Crypto.Signature import DSS
from Crypto.Hash import SHA256


class MiniBlockchain:
    """Mini-blockchain combinant toutes les primitives Cypherpunk."""

    def __init__(self, difficulty=3):
        self.chain = []
        self.difficulty = difficulty
        self.pending_transactions = []

    def create_genesis_block(self):
        """Creer le bloc genesis.
        TODO: Creer un bloc avec index=0, transactions=[], previous_hash="0"*64
        et le miner (trouver un nonce valide).
        """
        pass  # TODO: Implementez le bloc genesis


---

## 10. Resume

| Primitive | Inventeur | Annee | Role dans la blockchain |
|-----------|-----------|-------|------------------------|
| **Hash (SHA-256)** | NSA | 2001 | Integrite, adresses, PoW |
| **Chaine de hash** | Haber-Stornetta | 1991 | Immuabilite des blocs |
| **Arbre de Merkle** | Ralph Merkle | 1979 | Verification legere (SPV) |
| **Proof-of-Work** | Adam Back | 1997 | Consensus, anti-falsification |
| **Signatures ECDSA** | Johnson et al. | 2001 | Authentification des transactions |
| **DHT Kademlia** | Maymounkov-Mazieres | 2002 | Reseau decentralise P2P |

### Points cles

- Les smart contracts sont l'**heritage direct** du mouvement Cypherpunk des annees 1990
- Chaque brique (hash, signatures, PoW, P2P) existait **avant** Bitcoin
- L'innovation de Satoshi Nakamoto est la **combinaison** de ces primitives
- Comprendre ces fondations permet de mieux apprehender la securite et les limites des blockchains

---

**Notebook suivant** : [SC-1-Setup-Foundry](SC-1-Setup-Foundry.ipynb) - Installation de l'environnement de developpement

---

[Setup Foundry >>](SC-1-Setup-Foundry.ipynb)